Analyse de la Rétention : Comparaison par Traitement et Environnement (Inpatient/Outpatient)

In [ ]:
!pip install -U kaleido

In [ ]:
# On commence en chargeant les bibliothèques
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "colab"

###  1. CHARGEMENT ET PRÉPARATION DES GROUPES
dm1 = pd.read_csv("dm1.csv", dtype=str)
dm2 = pd.read_csv("dm2.csv", dtype=str)
dm = pd.concat([dm1, dm2], ignore_index=True)

## A. On filtre pour ne garder que les 2 médicaments (Pas de Screen Failure)
target_arms = ['CLONIDINE', 'BUPRENORPHINE/NALOXONE']
dm = dm[dm['ARM'].isin(target_arms)]

## B. On recrée notre variable IN / OUT (Selon l'étude)
# Si STUDYID contient '0001' c'est IN, sinon OUT
dm['Situation'] = np.where(dm['STUDYID'].str.contains("0001"), "Inpatient", "Outpatient")

## C. CRÉATION DES 4 GROUPES (C'est l'étape clé pour avoir 4 courbes)
# On combine le nom du médicament + la situation (ex: "CLONIDINE - Inpatient")
dm['Groupe_Analyse'] = dm['ARM'] + " (" + dm['Situation'] + ")"

print("Les 4 groupes qui seront affichés :")
print(dm['Groupe_Analyse'].unique())

###  2. TRAITEMENT DES VISITES (EX)
ex1 = pd.read_csv("ex1.csv", dtype=str)
ex2 = pd.read_csv("ex2.csv", dtype=str)
ex = pd.concat([ex1, ex2], ignore_index=True)

# On ajoute notre nouveau 'Groupe_Analyse' aux données de visites
ex = ex.merge(dm[['USUBJID', 'Groupe_Analyse']], on='USUBJID', how='inner')
ex['VISITNUM'] = pd.to_numeric(ex['VISITNUM'], errors='coerce')

###  3. CALCULS DE RÉTENTION (Nt / N0)

# N0 : Nombre de patients au départ par Groupe (4 groupes)
n0 = dm.groupby('Groupe_Analyse')['USUBJID'].nunique().reset_index(name='N0')
n0['VISITNUM'] = 0

# Nt : Nombre de patients restants à chaque visite
nt = ex.groupby(['Groupe_Analyse', 'VISITNUM'])['USUBJID'].nunique().reset_index(name='Nt')

# Fusion
df_plot = nt.merge(n0[['Groupe_Analyse', 'N0']], on='Groupe_Analyse', how='left')

# Ajout de la ligne de départ (100%)
start_rows = n0.copy()
start_rows['Nt'] = start_rows['N0']
df_final = pd.concat([start_rows, df_plot], ignore_index=True)

# Calcul du %
df_final['Taux_Retention'] = df_final['Nt'] / df_final['N0']

# Tri
df_final = df_final.sort_values(by=['Groupe_Analyse', 'VISITNUM'])

### 4. GRAPHIQUE
fig = px.line(
    df_final,
    x="VISITNUM",
    y="Taux_Retention",
    color="Groupe_Analyse", # Ceci va générer automatiquement les 4 couleurs
    markers=True,
    symbol="Groupe_Analyse", # Ajoute des formes différentes pour mieux distinguer
    title="Rétention par Traitement et par Situation (In/Out)",
    labels={
        "VISITNUM": "Semaines d'étude",
        "Taux_Retention": "Proportion de participants retenus",
        "Groupe_Analyse": "Groupe"
    }
)

fig.update_yaxes(range=[0, 1.05], tickformat=".0%")
fig.show()
fig.write_image("Figure1_Static.png", width=1200, height=600, scale=2)
print("Image statique 'Figure1_Static.png' générée avec succès.")

Les 4 groupes qui seront affichés :
['CLONIDINE (Inpatient)' 'BUPRENORPHINE/NALOXONE (Inpatient)'
 'BUPRENORPHINE/NALOXONE (Outpatient)' 'CLONIDINE (Outpatient)']


ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
